# Reranker Fine-Tuning for Turkish Legal RAG

This notebook fine-tunes the Turkish BGE reranker on legal retrieval pairs.

Previous best pre-fine-tuning pipeline:
- Source-aware retrieval
- Article-aware retrieval
- Turkish BGE reranker fusion
- Improved legal prompt

Best previous manual accuracy:
- 0.421053 on the 20-sample test evaluation
- 0.46875 on the coverage-clean subset

Goal:
- Improve reranking quality
- Increase the probability that the correct legal context appears in Top-3
- Evaluate whether a fine-tuned reranker improves retrieval performance

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU yok. Runtime > Change runtime type > T4 GPU seç.")

CUDA available: True
GPU: Tesla T4


In [3]:
project_path = "/content/drive/MyDrive/turkish_legal_rag"

data_path = f"{project_path}/data"
external_path = f"{data_path}/raw/external_datasets"
external_extracted_path = f"{external_path}/extracted"

outputs_path = f"{project_path}/outputs"
metrics_path = f"{outputs_path}/metrics"
models_path = f"{outputs_path}/models"

finetuned_reranker_path = f"{models_path}/turkish_bge_legal_reranker"

print("Project:", project_path)
print("External extracted:", external_extracted_path)
print("Metrics:", metrics_path)
print("Models:", models_path)
print("Fine-tuned reranker path:", finetuned_reranker_path)

Project: /content/drive/MyDrive/turkish_legal_rag
External extracted: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted
Metrics: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics
Models: /content/drive/MyDrive/turkish_legal_rag/outputs/models
Fine-tuned reranker path: /content/drive/MyDrive/turkish_legal_rag/outputs/models/turkish_bge_legal_reranker


In [4]:
!pip install -q -U sentence-transformers transformers accelerate datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 22.0 MB/s eta 0:00:00


In [5]:
import os
import re
import json
import glob
import random

import numpy as np
import pandas as pd
import torch

from tqdm import tqdm
from sklearn.model_selection import train_test_split

from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

In [6]:
reranker_candidates = glob.glob(
    f"{external_extracted_path}/**/reranker.jsonl",
    recursive=True
)

print("Reranker candidates:")
for path in reranker_candidates:
    print(path)

reranker_path = reranker_candidates[0]

print("Using reranker file:", reranker_path)

Reranker candidates:
/content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/reranker.jsonl
Using reranker file: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/reranker.jsonl


In [7]:
def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    return records


reranker_records = load_jsonl(reranker_path)

print("Reranker records:", len(reranker_records))
print("Sample record:")
reranker_records[0]

Reranker records: 6752
Sample record:


{'id': 'rerank_noleak_balanced_000001',
 'query_id': 'rerank_query_final_000001',
 'query': '18 yaşından küçük çocuk hakkında ORICON kaynağına göre ne söylenebilir?',
 'candidate_passage': '18 yaşından küçük bir çocuk için DNA testi yaptırılabilmesi için anne ve babanın rızasına ihtiyaç vardır.',
 'label': 1,
 'candidate_id': 'oricon_genel_001370',
 'citation_label': 'ORICON - Genel hukuk / sınıflandırma bekliyor - oricon_genel_001370',
 'source': 'ORICON',
 'negative_type': None,
 'audit_status': 'no_eval_gold_leak_balanced_v3'}

In [8]:
print("First 5 records keys:")

for i, rec in enumerate(reranker_records[:5]):
    print("=" * 80)
    print("Index:", i)
    print("Keys:", rec.keys())
    print(rec)

First 5 records keys:
Index: 0
Keys: dict_keys(['id', 'query_id', 'query', 'candidate_passage', 'label', 'candidate_id', 'citation_label', 'source', 'negative_type', 'audit_status'])
{'id': 'rerank_noleak_balanced_000001', 'query_id': 'rerank_query_final_000001', 'query': '18 yaşından küçük çocuk hakkında ORICON kaynağına göre ne söylenebilir?', 'candidate_passage': '18 yaşından küçük bir çocuk için DNA testi yaptırılabilmesi için anne ve babanın rızasına ihtiyaç vardır.', 'label': 1, 'candidate_id': 'oricon_genel_001370', 'citation_label': 'ORICON - Genel hukuk / sınıflandırma bekliyor - oricon_genel_001370', 'source': 'ORICON', 'negative_type': None, 'audit_status': 'no_eval_gold_leak_balanced_v3'}
Index: 1
Keys: dict_keys(['id', 'query_id', 'query', 'candidate_passage', 'label', 'candidate_id', 'citation_label', 'source', 'negative_type', 'audit_status'])
{'id': 'rerank_noleak_balanced_000002', 'query_id': 'rerank_query_final_000001', 'query': '18 yaşından küçük çocuk hakkında ORICO

In [9]:
def normalize_reranker_records(records):
    rows = []

    for rec in records:
        query = rec.get("query")
        passage = rec.get("candidate_passage")
        label = rec.get("label")

        if query is not None and passage is not None and label is not None:
            rows.append({
                "query": str(query),
                "passage": str(passage),
                "label": float(label),
                "query_id": rec.get("query_id"),
                "candidate_id": rec.get("candidate_id"),
                "source": rec.get("source"),
                "negative_type": rec.get("negative_type"),
                "audit_status": rec.get("audit_status")
            })

    return pd.DataFrame(rows)


reranker_df = normalize_reranker_records(reranker_records)

print("Normalized reranker pairs:", reranker_df.shape)
display(reranker_df.head())

print("Label distribution:")
display(reranker_df["label"].value_counts())

Normalized reranker pairs: (6752, 8)


,query,passage,label,query_id,candidate_id,source,negative_type,audit_status
0,18 yaşından küçük çocuk hakkında ORICON kaynağ...,18 yaşından küçük bir çocuk için DNA testi yap...,1.0,rerank_query_final_000001,oricon_genel_001370,ORICON,None,no_eval_gold_leak_balanced_v3
1,18 yaşından küçük çocuk hakkında ORICON kaynağ...,İdarenin takdir yetkisi kamu yararı ve hizmet ...,0.0,rerank_query_final_000001,oricon_genel_000975,ORICON,hard_negative_same_source_or_category,no_eval_gold_leak_balanced_v3
2,18 yaşından küçük çocuk hakkında ORICON kaynağ...,Satış bedelinin tamamının peşin ödenmesi hâlin...,0.0,rerank_query_final_000001,oricon_genel_001651,ORICON,hard_negative_same_source_or_category,no_eval_gold_leak_balanced_v3
3,18 yaşını dolduran çocuk eğitimine hakkında OR...,18 yaşını dolduran çocuk eğitimine devam ediyo...,1.0,rerank_query_final_000002,oricon_genel_001778,ORICON,None,no_eval_gold_leak_balanced_v3
4,18 yaşını dolduran çocuk eğitimine hakkında OR...,"İtiraz, Sulh Ceza Hakimliğine yapılmalıdır ve ...",0.0,rerank_query_final_000002,oricon_genel_000063,ORICON,hard_negative_same_source_or_category,no_eval_gold_leak_balanced_v3


Label distribution:


,count
label,
0.0,4299
1.0,2453


In [10]:
reranker_df = reranker_df.dropna(subset=["query", "passage", "label"]).reset_index(drop=True)

reranker_df["query"] = reranker_df["query"].astype(str)
reranker_df["passage"] = reranker_df["passage"].astype(str)
reranker_df["label"] = reranker_df["label"].astype(float)

reranker_df["label"] = (reranker_df["label"] > 0).astype(float)

reranker_df = reranker_df[
    reranker_df["query"].str.len() > 3
].copy()

reranker_df = reranker_df[
    reranker_df["passage"].str.len() > 10
].copy()

reranker_df = reranker_df.reset_index(drop=True)

print("Clean reranker pairs:", reranker_df.shape)

print("Label distribution:")
display(reranker_df["label"].value_counts())

display(reranker_df.sample(5, random_state=42))

Clean reranker pairs: (6752, 8)
Label distribution:


,count
label,
0.0,4299
1.0,2453


,query,passage,label,query_id,candidate_id,source,negative_type,audit_status
6269,Unutulma hakkı hakkında ORICON kaynağına göre ...,"Yapı kayıt belgesi başvuru bedeli, arsa emlak ...",0.0,rerank_query_final_001902,oricon_ticaret_000377,ORICON,hard_negative_same_source_or_category,no_eval_gold_leak_balanced_v3
1410,Eşin hırsız namussuz olduğuna dair hakkında OR...,Eşin hırsız veya namussuz olduğuna dair söylem...,1.0,rerank_query_final_000503,oricon_genel_001819,ORICON,None,no_eval_gold_leak_balanced_v3
1554,grev hakkında ORICON kaynağına göre ne söylene...,"Grev engelleri, grev oylaması, grev ertelemesi...",1.0,rerank_query_final_000550,oricon_genel_000864,ORICON,None,no_eval_gold_leak_balanced_v3
5169,Medeni Hukuk alanında IV. Yükten kurtarma hakk...,"Madde 801- İntifa hakkı sahibi, yükümlü olmadı...",0.0,rerank_query_final_001662,turkish_law_eski_4721_turk_medeni_kanunu_m801,TURKISH_LAW_ESKI_LOW_RISK_ONLY,hard_negative_same_source_or_category,no_eval_gold_leak_balanced_v3
2350,"Hukuk Genel Kurulu 2015/167 E., 2015/1328 K. s...","İçtihat Metni "" Taraflar arasındaki “evlenmeni...",1.0,rerank_query_final_000705,yargitay_1274_aile_hukuku_003,YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY,None,no_eval_gold_leak_balanced_v3


In [11]:
from sklearn.model_selection import train_test_split

print("Unique query count:", reranker_df["query_id"].nunique())
print("Total pairs:", len(reranker_df))

display(reranker_df["query_id"].value_counts().head())

Unique query count: 1689
Total pairs: 6752


,count
query_id,
rerank_query_final_001798,112
rerank_query_final_000286,84
rerank_query_final_001689,79
rerank_query_final_000849,65
rerank_query_final_000839,57


In [12]:
unique_query_ids = reranker_df["query_id"].dropna().unique()

train_query_ids, val_query_ids = train_test_split(
    unique_query_ids,
    test_size=0.1,
    random_state=42
)

train_pairs_df = reranker_df[
    reranker_df["query_id"].isin(train_query_ids)
].reset_index(drop=True)

val_pairs_df = reranker_df[
    reranker_df["query_id"].isin(val_query_ids)
].reset_index(drop=True)

print("Train pairs:", train_pairs_df.shape)
print("Val pairs:", val_pairs_df.shape)

print("\nTrain label distribution:")
display(train_pairs_df["label"].value_counts())

print("\nVal label distribution:")
display(val_pairs_df["label"].value_counts())

Train pairs: (6072, 8)
Val pairs: (680, 8)

Train label distribution:


,count
label,
0.0,3859
1.0,2213



Val label distribution:


,count
label,
0.0,440
1.0,240


In [13]:
from sentence_transformers import InputExample

train_examples = [
    InputExample(
        texts=[row["query"], row["passage"]],
        label=float(row["label"])
    )
    for _, row in train_pairs_df.iterrows()
]

val_examples = [
    InputExample(
        texts=[row["query"], row["passage"]],
        label=float(row["label"])
    )
    for _, row in val_pairs_df.iterrows()
]

print("Train examples:", len(train_examples))
print("Val examples:", len(val_examples))

Train examples: 6072
Val examples: 680


In [14]:
base_reranker_model_name = "seroe/bge-reranker-v2-m3-turkish-triplet"

reranker_model = CrossEncoder(
    base_reranker_model_name,
    num_labels=1,
    max_length=512,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Loaded base reranker:", base_reranker_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Loaded base reranker: seroe/bge-reranker-v2-m3-turkish-triplet


In [15]:
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

val_sentence_pairs = list(
    zip(
        val_pairs_df["query"].tolist(),
        val_pairs_df["passage"].tolist()
    )
)

val_labels = val_pairs_df["label"].astype(float).tolist()

evaluator = CEBinaryClassificationEvaluator(
    sentence_pairs=val_sentence_pairs,
    labels=val_labels,
    name="legal-reranker-val"
)

print("Evaluator ready.")

Evaluator ready.


/tmp/ipykernel_2157/674204697.py:12: DeprecationWarning: This evaluator has been deprecated in favor of the more general CrossEncoderClassificationEvaluator. Please use CrossEncoderClassificationEvaluator instead, which supports both binary and multi-class evaluation. It accepts approximately the same inputs as this evaluator.
  evaluator = CEBinaryClassificationEvaluator(


In [16]:
from torch.utils.data import DataLoader

os.makedirs(finetuned_reranker_path, exist_ok=True)

train_batch_size = 4
num_epochs = 1

warmup_steps = max(
    10,
    int(len(train_examples) * num_epochs / train_batch_size * 0.1)
)

print("Train batch size:", train_batch_size)
print("Epochs:", num_epochs)
print("Warmup steps:", warmup_steps)
print("Output path:", finetuned_reranker_path)

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=train_batch_size
)

reranker_model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    output_path=finetuned_reranker_path,
    save_best_model=True,
    show_progress_bar=True
)

print("Fine-tuning completed.")

Train batch size: 4
Epochs: 1
Warmup steps: 151
Output path: /content/drive/MyDrive/turkish_legal_rag/outputs/models/turkish_bge_legal_reranker


Step,Training Loss
500,0.388614
1000,0.353135
1500,0.315203


Fine-tuning completed.


In [19]:
import os
import shutil

bad_path = finetuned_reranker_path + "_eval_only_save"

# Daha önce aynı yedek varsa sil
if os.path.exists(bad_path):
    shutil.rmtree(bad_path)

# Sadece eval içeren klasörü yedekle
if os.path.exists(finetuned_reranker_path):
    shutil.move(finetuned_reranker_path, bad_path)

# Yeni temiz model klasörü oluştur
os.makedirs(finetuned_reranker_path, exist_ok=True)

# Eğitilmiş CrossEncoder modelini manuel kaydet
reranker_model.save(finetuned_reranker_path)

print("Model manually saved to:", finetuned_reranker_path)
print("Files after manual save:")

for item in os.listdir(finetuned_reranker_path):
    print("-", item)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model manually saved to: /content/drive/MyDrive/turkish_legal_rag/outputs/models/turkish_bge_legal_reranker
Files after manual save:
- config_sentence_transformers.json
- config.json
- model.safetensors
- tokenizer_config.json
- tokenizer.json
- sentence_bert_config.json
- modules.json
- README.md


In [21]:
finetuned_reranker = CrossEncoder(
    finetuned_reranker_path,
    max_length=512,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Fine-tuned reranker loaded from:", finetuned_reranker_path)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Fine-tuned reranker loaded from: /content/drive/MyDrive/turkish_legal_rag/outputs/models/turkish_bge_legal_reranker


In [22]:
sample_val = val_pairs_df.sample(n=10, random_state=42).reset_index(drop=True)

pairs = sample_val[["query", "passage"]].values.tolist()

scores = finetuned_reranker.predict(
    pairs,
    batch_size=4,
    show_progress_bar=False
)

sample_val["pred_score"] = scores

display(sample_val[[
    "query",
    "label",
    "pred_score",
    "source",
    "passage"
]])

,query,label,pred_score,source,passage
0,Yeni görev yerinde iş başı hakkında ORICON kay...,1.0,0.999611,ORICON,"Yeni görev yerinde iş başı yapmayan bir memur,..."
1,ticaret hakkında ORICON kaynağına göre ne söyl...,1.0,0.904980,ORICON,"Konkordato talep edilen kişi tacir ise, başvur..."
2,Ceza Muhakemesi Hukuku alanında ceza muhakemes...,0.0,0.005752,TURKISH_LAW_ESKI_LOW_RISK_ONLY,"Madde 202 – (1) Sanık veya mağdur, meramını an..."
3,"Hukuk Genel Kurulu 2025/69 E., 2025/13 K. sayı...",0.0,0.000199,YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY,Kaynak türü: Yargıtay Hukuk Genel Kurulu karar...
4,Ceza Muhakemesi Hukuku alanında Yetkisizlik id...,0.0,0.001229,TURKISH_LAW_ESKI_LOW_RISK_ONLY,"Madde 172 – (1) Cumhuriyet savcısı, soruşturma..."
5,Medeni Hukuk alanında VIII. Ret hâlinde soruml...,0.0,0.000206,TURKISH_LAW_ESKI_LOW_RISK_ONLY,"Madde 275 - Mal rejimi sona erince, mevcut ort..."
6,"Hukuk Genel Kurulu 2023/948 E., 2024/256 K. sa...",0.0,0.005477,YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY,2. Değerlendirme 1. Davaya vekâlet 6100 sayılı...
7,Gemi ipoteği hakkında ORICON kaynağına göre ne...,1.0,0.999602,ORICON,"Gemi ipoteği, alacak hakkına sıkı sıkıya bağlı..."
8,yürütmenin durdurulması hakkında ORICON kaynağ...,0.0,0.000195,ORICON,"İptal davalarında idari işlemin yetki, şekil, ..."
9,Medeni Hukuk alanında A. Konusu hakkında kayna...,0.0,0.012386,TURKISH_LAW_ESKI_LOW_RISK_ONLY,"Madde 12 - Onbeş yaşını dolduran küçük, kendi ..."


In [23]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

val_pairs = val_pairs_df[["query", "passage"]].values.tolist()

val_scores = finetuned_reranker.predict(
    val_pairs,
    batch_size=8,
    show_progress_bar=True
)

val_pairs_df["pred_score"] = val_scores

# CrossEncoder skorları probability olmayabilir.
# Basit threshold olarak 0 kullanıyoruz.
val_pairs_df["pred_label"] = (val_pairs_df["pred_score"] > 0).astype(float)

accuracy = accuracy_score(
    val_pairs_df["label"],
    val_pairs_df["pred_label"]
)

auc = roc_auc_score(
    val_pairs_df["label"],
    val_pairs_df["pred_score"]
)

print("Validation accuracy:", accuracy)
print("Validation AUC:", auc)

print(classification_report(
    val_pairs_df["label"],
    val_pairs_df["pred_label"]
))

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Validation accuracy: 0.35294117647058826
Validation AUC: 0.9884564393939393
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       440
         1.0       0.35      1.00      0.52       240

    accuracy                           0.35       680
   macro avg       0.18      0.50      0.26       680
weighted avg       0.12      0.35      0.18       680



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [24]:
base_reranker_for_eval = CrossEncoder(
    base_reranker_model_name,
    max_length=512,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

base_val_scores = base_reranker_for_eval.predict(
    val_pairs,
    batch_size=8,
    show_progress_bar=True
)

val_pairs_df["base_pred_score"] = base_val_scores
val_pairs_df["base_pred_label"] = (val_pairs_df["base_pred_score"] > 0).astype(float)

base_accuracy = accuracy_score(
    val_pairs_df["label"],
    val_pairs_df["base_pred_label"]
)

base_auc = roc_auc_score(
    val_pairs_df["label"],
    val_pairs_df["base_pred_score"]
)

print("Base Validation accuracy:", base_accuracy)
print("Base Validation AUC:", base_auc)

print("Fine-tuned Validation accuracy:", accuracy)
print("Fine-tuned Validation AUC:", auc)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Base Validation accuracy: 0.35294117647058826
Base Validation AUC: 0.8386268939393942
Fine-tuned Validation accuracy: 0.35294117647058826
Fine-tuned Validation AUC: 0.9884564393939393


In [25]:
finetune_summary_df = pd.DataFrame([{
    "base_model": base_reranker_model_name,
    "training_dataset": "ceng493_starlar_reranker_jsonl",
    "train_pairs": len(train_pairs_df),
    "val_pairs": len(val_pairs_df),
    "train_batch_size": train_batch_size,
    "num_epochs": num_epochs,
    "warmup_steps": warmup_steps,
    "base_validation_accuracy_threshold_0": base_accuracy,
    "base_validation_auc": base_auc,
    "finetuned_validation_accuracy_threshold_0": accuracy,
    "finetuned_validation_auc": auc,
    "output_path": finetuned_reranker_path
}])

finetune_summary_df

,base_model,training_dataset,train_pairs,val_pairs,train_batch_size,num_epochs,warmup_steps,base_validation_accuracy_threshold_0,base_validation_auc,finetuned_validation_accuracy_threshold_0,finetuned_validation_auc,output_path
0,seroe/bge-reranker-v2-m3-turkish-triplet,ceng493_starlar_reranker_jsonl,6072,680,4,1,151,0.352941,0.838627,0.352941,0.988456,/content/drive/MyDrive/turkish_legal_rag/outpu...


In [26]:
finetune_summary_df.to_csv(
    f"{metrics_path}/finetuned_turkish_bge_reranker_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

val_pairs_df.to_csv(
    f"{metrics_path}/finetuned_turkish_bge_reranker_val_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Fine-tuned reranker metrics saved.")

Fine-tuned reranker metrics saved.


In [27]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def find_best_threshold(y_true, scores):
    thresholds = np.linspace(scores.min(), scores.max(), 200)

    best = {
        "threshold": None,
        "accuracy": -1,
        "precision": None,
        "recall": None,
        "f1": -1
    }

    for threshold in thresholds:
        preds = (scores >= threshold).astype(float)

        accuracy = accuracy_score(y_true, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            preds,
            average="binary",
            zero_division=0
        )

        if f1 > best["f1"]:
            best = {
                "threshold": threshold,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1
            }

    return best


y_true = val_pairs_df["label"].values.astype(float)

base_scores = val_pairs_df["base_pred_score"].values
finetuned_scores = val_pairs_df["pred_score"].values

base_best = find_best_threshold(y_true, base_scores)
finetuned_best = find_best_threshold(y_true, finetuned_scores)

print("Base AUC:", roc_auc_score(y_true, base_scores))
print("Base best threshold metrics:")
print(base_best)

print("\nFine-tuned AUC:", roc_auc_score(y_true, finetuned_scores))
print("Fine-tuned best threshold metrics:")
print(finetuned_best)

Base AUC: 0.8386268939393942
Base best threshold metrics:
{'threshold': np.float32(0.005037543), 'accuracy': 0.7573529411764706, 'precision': 0.631578947368421, 'recall': 0.75, 'f1': 0.6857142857142857}

Fine-tuned AUC: 0.9884564393939393
Fine-tuned best threshold metrics:
{'threshold': np.float32(0.7283975), 'accuracy': 0.9514705882352941, 'precision': 0.9330543933054394, 'recall': 0.9291666666666667, 'f1': 0.9311064718162839}


In [28]:
finetune_summary_df = pd.DataFrame([{
    "base_model": base_reranker_model_name,
    "training_dataset": "ceng493_starlar_reranker_jsonl",
    "train_pairs": len(train_pairs_df),
    "val_pairs": len(val_pairs_df),
    "train_batch_size": train_batch_size,
    "num_epochs": num_epochs,
    "warmup_steps": warmup_steps,

    "base_validation_auc": base_auc,
    "base_best_threshold": base_best["threshold"],
    "base_best_threshold_accuracy": base_best["accuracy"],
    "base_best_threshold_precision": base_best["precision"],
    "base_best_threshold_recall": base_best["recall"],
    "base_best_threshold_f1": base_best["f1"],

    "finetuned_validation_auc": auc,
    "finetuned_best_threshold": finetuned_best["threshold"],
    "finetuned_best_threshold_accuracy": finetuned_best["accuracy"],
    "finetuned_best_threshold_precision": finetuned_best["precision"],
    "finetuned_best_threshold_recall": finetuned_best["recall"],
    "finetuned_best_threshold_f1": finetuned_best["f1"],

    "output_path": finetuned_reranker_path
}])

finetune_summary_df

,base_model,training_dataset,train_pairs,val_pairs,train_batch_size,num_epochs,warmup_steps,base_validation_auc,base_best_threshold,base_best_threshold_accuracy,base_best_threshold_precision,base_best_threshold_recall,base_best_threshold_f1,finetuned_validation_auc,finetuned_best_threshold,finetuned_best_threshold_accuracy,finetuned_best_threshold_precision,finetuned_best_threshold_recall,finetuned_best_threshold_f1,output_path
0,seroe/bge-reranker-v2-m3-turkish-triplet,ceng493_starlar_reranker_jsonl,6072,680,4,1,151,0.838627,0.005038,0.757353,0.631579,0.75,0.685714,0.988456,0.728397,0.951471,0.933054,0.929167,0.931106,/content/drive/MyDrive/turkish_legal_rag/outpu...


In [29]:
finetune_summary_df.to_csv(
    f"{metrics_path}/finetuned_turkish_bge_reranker_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

val_pairs_df.to_csv(
    f"{metrics_path}/finetuned_turkish_bge_reranker_val_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Fine-tuned reranker metrics saved.")

Fine-tuned reranker metrics saved.


## Retrieval Evaluation with Fine-Tuned Reranker

This section evaluates whether the fine-tuned reranker improves retrieval ranking on the CENG493 Starlar RAG evaluation set.

We compare:
- Hybrid Retrieval
- Hybrid Retrieval + Base Turkish BGE Reranker
- Hybrid Retrieval + Fine-tuned Turkish BGE Reranker

The goal is to check whether the fine-tuned reranker can move gold chunks higher in the ranking.

In [30]:
starlar_corpus_candidates = glob.glob(
    f"{external_extracted_path}/**/corpus.jsonl",
    recursive=True
)

starlar_rag_eval_candidates = glob.glob(
    f"{external_extracted_path}/**/rag_eval.json",
    recursive=True
)

print("Starlar corpus candidates:")
for p in starlar_corpus_candidates:
    print(p)

print("\nStarlar rag_eval candidates:")
for p in starlar_rag_eval_candidates:
    print(p)

starlar_corpus_path = starlar_corpus_candidates[0]
starlar_rag_eval_path = starlar_rag_eval_candidates[0]

print("\nUsing corpus:", starlar_corpus_path)
print("Using rag_eval:", starlar_rag_eval_path)

Starlar corpus candidates:
/content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/corpus.jsonl

Starlar rag_eval candidates:
/content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/rag_eval.json

Using corpus: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/corpus.jsonl
Using rag_eval: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/rag_eval.json


In [31]:
def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    return records


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [32]:
starlar_corpus_records = load_jsonl(starlar_corpus_path)
starlar_rag_eval_records = load_json(starlar_rag_eval_path)

print("Starlar corpus records:", len(starlar_corpus_records))
print("Starlar rag_eval records:", len(starlar_rag_eval_records))

print("\nSample corpus record:")
print(starlar_corpus_records[0])

print("\nSample rag_eval record:")
print(starlar_rag_eval_records[0])

Starlar corpus records: 7579
Starlar rag_eval records: 1000

Sample corpus record:
{'id': 'oricon_anayasa_000001', 'text': 'Susma hakkı, kişinin kendi lehine veya aleyhine ifade verme kararını özgürce kullanabilmesine olanak tanır.', 'title': 'Anayasa Hukuku ve Temel Haklar', 'metadata': {'source': 'ORICON', 'source_file': 'ORICON_clean_legal_source_for_chunking.txt', 'doc_id': 'oricon_legal_source', 'chunk_id': 'oricon_anayasa_000001', 'chunk_index': 1, 'category_chunk_index': 1, 'category': 'Anayasa Hukuku ve Temel Haklar', 'category_slug': 'anayasa', 'semantic_topic': 'susma hakkı', 'legal_concepts': ['susma hakkı'], 'citation_label': 'ORICON - Anayasa Hukuku ve Temel Haklar - oricon_anayasa_000001', 'verification_required': False, 'original_flags': None, 'quality_flags': None, 'source_location': {'line_start': 11, 'line_end': 11, 'char_start': 370, 'char_end': 477}, 'source_record_count': 1, 'chunk_token_count_approx': 16, 'chunk_char_count': 107, 'language': 'tr', 'previous_chunk_

In [33]:
starlar_rows = []

for record in starlar_corpus_records:
    metadata = record.get("metadata", {}) or {}

    starlar_rows.append({
        "chunk_id": record.get("id"),
        "text": record.get("text"),
        "title": record.get("title"),
        "source": metadata.get("source"),
        "category": metadata.get("category"),
        "citation_label": metadata.get("citation_label")
    })

starlar_df = pd.DataFrame(starlar_rows)

starlar_df = starlar_df.dropna(subset=["text", "chunk_id"]).reset_index(drop=True)

starlar_df["chunk_id"] = starlar_df["chunk_id"].astype(str)
starlar_df["source"] = starlar_df["source"].astype(str)
starlar_df["text"] = starlar_df["text"].astype(str)

print("Starlar corpus df:", starlar_df.shape)
display(starlar_df.head())

print("Source counts:")
display(starlar_df["source"].value_counts().head(20))

Starlar corpus df: (7579, 6)


,chunk_id,text,title,source,category,citation_label
0,oricon_anayasa_000001,"Susma hakkı, kişinin kendi lehine veya aleyhin...",Anayasa Hukuku ve Temel Haklar,ORICON,Anayasa Hukuku ve Temel Haklar,ORICON - Anayasa Hukuku ve Temel Haklar - oric...
1,oricon_anayasa_000003,"Düşünce özgürlüğü, düşünce ve kanaatlerin çeşi...",Anayasa Hukuku ve Temel Haklar,ORICON,Anayasa Hukuku ve Temel Haklar,ORICON - Anayasa Hukuku ve Temel Haklar - oric...
2,oricon_anayasa_000004,"İfade özgürlüğü, demokratik toplumların temeli...",Anayasa Hukuku ve Temel Haklar,ORICON,Anayasa Hukuku ve Temel Haklar,ORICON - Anayasa Hukuku ve Temel Haklar - oric...
3,oricon_anayasa_000005,Anayasada düzenlenen temel hak ve özgürlüklerd...,Anayasa Hukuku ve Temel Haklar,ORICON,Anayasa Hukuku ve Temel Haklar,ORICON - Anayasa Hukuku ve Temel Haklar - oric...
4,oricon_anayasa_000006,Demokratik bir toplumda eleştiri hakkının koru...,Anayasa Hukuku ve Temel Haklar,ORICON,Anayasa Hukuku ve Temel Haklar,ORICON - Anayasa Hukuku ve Temel Haklar - oric...


Source counts:


,count
source,
ORICON,3742
TURKISH_LAW_ESKI_LOW_RISK_ONLY,1727
YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY,1242
TRAIN_LOW_RISK_ONLY,538
TURKISH_LAWCHATBOT_LOW_RISK_ONLY,330


In [34]:
def simple_turkish_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)
    tokens = text.split()

    stopwords = {
        "ve", "veya", "ile", "de", "da", "bir", "bu", "şu", "o",
        "için", "gibi", "olarak", "olan", "kadar", "ise", "ancak",
        "çok", "daha", "en", "mi", "mı", "mu", "mü"
    }

    return [t for t in tokens if t not in stopwords and len(t) > 1]


def min_max_normalize(scores):
    scores = np.array(scores, dtype=np.float32)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (scores - scores.min()) / (scores.max() - scores.min())

In [36]:
!pip install -q -U sentence-transformers faiss-cpu rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 76.8 MB/s eta 0:00:00


In [37]:
import re
import json
import glob
import numpy as np
import pandas as pd
import faiss

from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

In [38]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(embedding_model_name)

starlar_texts = starlar_df["text"].astype(str).tolist()

starlar_embeddings = embedding_model.encode(
    starlar_texts,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

faiss.normalize_L2(starlar_embeddings)

starlar_index = faiss.IndexFlatIP(starlar_embeddings.shape[1])
starlar_index.add(starlar_embeddings)

tokenized_starlar_corpus = [
    simple_turkish_tokenize(text)
    for text in starlar_texts
]

starlar_bm25 = BM25Okapi(tokenized_starlar_corpus)

print("Starlar FAISS vectors:", starlar_index.ntotal)
print("BM25 ready.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/237 [00:00<?, ?it/s]

Starlar FAISS vectors: 7579
BM25 ready.


In [39]:
def starlar_hybrid_retrieve_candidates(query, candidate_k=20, alpha=0.5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    dense_scores, dense_indices = starlar_index.search(
        query_embedding,
        len(starlar_df)
    )

    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]

    dense_score_map = {
        int(idx): float(score)
        for idx, score in zip(dense_indices, dense_scores)
    }

    dense_all_scores = np.array([
        dense_score_map.get(i, 0.0)
        for i in range(len(starlar_df))
    ])

    bm25_scores = np.array(
        starlar_bm25.get_scores(simple_turkish_tokenize(query))
    )

    dense_norm = min_max_normalize(dense_all_scores)
    bm25_norm = min_max_normalize(bm25_scores)

    final_scores = alpha * dense_norm + (1 - alpha) * bm25_norm

    top_indices = np.argsort(final_scores)[::-1][:candidate_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "chunk_id": str(starlar_df.iloc[idx]["chunk_id"]),
            "source": str(starlar_df.iloc[idx]["source"]),
            "hybrid_score": float(final_scores[idx]),
            "text": starlar_df.iloc[idx]["text"]
        })

    return results

In [40]:
def rerank_candidates(query, candidates, reranker, top_k=5, batch_size=8):
    pairs = [
        [query, item["text"]]
        for item in candidates
    ]

    scores = reranker.predict(
        pairs,
        batch_size=batch_size,
        show_progress_bar=False
    )

    reranked = []

    for item, score in zip(candidates, scores):
        copied = item.copy()
        copied["rerank_score"] = float(score)
        reranked.append(copied)

    reranked = sorted(
        reranked,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:top_k]

In [41]:
base_reranker_for_eval = CrossEncoder(
    base_reranker_model_name,
    max_length=512,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Base reranker loaded for retrieval evaluation.")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Base reranker loaded for retrieval evaluation.


In [42]:
sample_eval = pd.DataFrame(starlar_rag_eval_records).sample(
    n=1,
    random_state=42
).iloc[0]

query = sample_eval["query"]
gold_chunk_ids = [str(x) for x in sample_eval["gold_chunk_ids"]]

candidates = starlar_hybrid_retrieve_candidates(
    query,
    candidate_k=20,
    alpha=0.5
)

hybrid_top5 = candidates[:5]

base_top5 = rerank_candidates(
    query=query,
    candidates=candidates,
    reranker=base_reranker_for_eval,
    top_k=5,
    batch_size=8
)

finetuned_top5 = rerank_candidates(
    query=query,
    candidates=candidates,
    reranker=finetuned_reranker,
    top_k=5,
    batch_size=8
)

print("QUERY:", query)
print("GOLD:", gold_chunk_ids)

print("\nHYBRID TOP5:")
for r in hybrid_top5:
    print(r["rank"], r["chunk_id"], r["source"], r.get("hybrid_score"))

print("\nBASE RERANK TOP5:")
for i, r in enumerate(base_top5, start=1):
    print(i, r["chunk_id"], r["source"], r.get("rerank_score"))

print("\nFINETUNED RERANK TOP5:")
for i, r in enumerate(finetuned_top5, start=1):
    print(i, r["chunk_id"], r["source"], r.get("rerank_score"))

QUERY: Türk Medenî Kanunu madde 76 kapsamında BİRİNCİ KİTAP konusunda Madde Bütün üyelerin araya gelmeksizin yazılı katılımıyla alınan düzenlemesi nasıl açıklanır?
GOLD: ['turkish_law_eski_4721_turk_medeni_kanunu_m76']

HYBRID TOP5:
1 turkish_law_eski_4721_turk_medeni_kanunu_m76 TURKISH_LAW_ESKI_LOW_RISK_ONLY 0.9452148675918579
2 turkish_law_eski_5271_ceza_muhakemesi_kanunu_m243 TURKISH_LAW_ESKI_LOW_RISK_ONLY 0.6747979521751404
3 turkish_law_eski_5271_ceza_muhakemesi_kanunu_m259 TURKISH_LAW_ESKI_LOW_RISK_ONLY 0.6680404543876648
4 turkish_law_eski_5271_ceza_muhakemesi_kanunu_m232_p002 TURKISH_LAW_ESKI_LOW_RISK_ONLY 0.6593407988548279
5 turkish_law_eski_4721_turk_medeni_kanunu_m431 TURKISH_LAW_ESKI_LOW_RISK_ONLY 0.5788350105285645

BASE RERANK TOP5:
1 turkish_law_eski_4721_turk_medeni_kanunu_m76 TURKISH_LAW_ESKI_LOW_RISK_ONLY 0.9641590714454651
2 turkish_law_eski_5271_ceza_muhakemesi_kanunu_m243 TURKISH_LAW_ESKI_LOW_RISK_ONLY 0.009145756252110004
3 turkish_law_eski_4721_turk_medeni_kanun

In [43]:
starlar_eval_df = pd.DataFrame(starlar_rag_eval_records).sample(
    n=min(100, len(starlar_rag_eval_records)),
    random_state=42
).reset_index(drop=True)

candidate_k = 20
final_k = 5

comparison_rows = []

for i, row in tqdm(starlar_eval_df.iterrows(), total=len(starlar_eval_df)):
    query = row["query"]
    gold_chunk_ids = [str(x) for x in row["gold_chunk_ids"]]

    candidates = starlar_hybrid_retrieve_candidates(
        query,
        candidate_k=candidate_k,
        alpha=0.5
    )

    hybrid_top5 = candidates[:final_k]

    base_top5 = rerank_candidates(
        query=query,
        candidates=candidates,
        reranker=base_reranker_for_eval,
        top_k=final_k,
        batch_size=8
    )

    finetuned_top5 = rerank_candidates(
        query=query,
        candidates=candidates,
        reranker=finetuned_reranker,
        top_k=final_k,
        batch_size=8
    )

    hybrid_ids = [str(x["chunk_id"]) for x in hybrid_top5]
    base_ids = [str(x["chunk_id"]) for x in base_top5]
    finetuned_ids = [str(x["chunk_id"]) for x in finetuned_top5]

    comparison_rows.append({
        "index": i,
        "query": query,
        "gold_chunk_ids": "; ".join(gold_chunk_ids),

        "hybrid_top1": hybrid_ids[0],
        "base_rerank_top1": base_ids[0],
        "finetuned_rerank_top1": finetuned_ids[0],

        "hybrid_top5": "; ".join(hybrid_ids),
        "base_rerank_top5": "; ".join(base_ids),
        "finetuned_rerank_top5": "; ".join(finetuned_ids),

        "hybrid_hit_at_1": any(gold_id == hybrid_ids[0] for gold_id in gold_chunk_ids),
        "hybrid_hit_at_5": any(gold_id in hybrid_ids for gold_id in gold_chunk_ids),

        "base_rerank_hit_at_1": any(gold_id == base_ids[0] for gold_id in gold_chunk_ids),
        "base_rerank_hit_at_5": any(gold_id in base_ids for gold_id in gold_chunk_ids),

        "finetuned_rerank_hit_at_1": any(gold_id == finetuned_ids[0] for gold_id in gold_chunk_ids),
        "finetuned_rerank_hit_at_5": any(gold_id in finetuned_ids for gold_id in gold_chunk_ids),
    })

reranker_retrieval_comparison_df = pd.DataFrame(comparison_rows)

reranker_retrieval_comparison_df.head()

100%|██████████| 100/100 [04:26<00:00,  2.66s/it]


,index,query,gold_chunk_ids,hybrid_top1,base_rerank_top1,finetuned_rerank_top1,hybrid_top5,base_rerank_top5,finetuned_rerank_top5,hybrid_hit_at_1,hybrid_hit_at_5,base_rerank_hit_at_1,base_rerank_hit_at_5,finetuned_rerank_hit_at_1,finetuned_rerank_hit_at_5
0,0,Türk Medenî Kanunu madde 76 kapsamında BİRİNCİ...,turkish_law_eski_4721_turk_medeni_kanunu_m76,turkish_law_eski_4721_turk_medeni_kanunu_m76,turkish_law_eski_4721_turk_medeni_kanunu_m76,turkish_law_eski_4721_turk_medeni_kanunu_m76,turkish_law_eski_4721_turk_medeni_kanunu_m76; ...,turkish_law_eski_4721_turk_medeni_kanunu_m76; ...,turkish_law_eski_4721_turk_medeni_kanunu_m76; ...,True,True,True,True,True,True
1,1,"Hukuk Genel Kurulu 2015/231 E., 2015/1467 K. s...",yargitay_1273_aile_hukuku_004,yargitay_1273_aile_hukuku_001,yargitay_1273_aile_hukuku_001,yargitay_1273_aile_hukuku_004,yargitay_1273_aile_hukuku_001; yargitay_1344_a...,yargitay_1273_aile_hukuku_001; yargitay_1273_a...,yargitay_1273_aile_hukuku_004; yargitay_1273_a...,False,False,False,True,True,True
2,2,"Hukuk Genel Kurulu 2013/1910 E., 2015/1203 K. ...",yargitay_0949_icra_ve_iflas_hukuku_004,yargitay_0949_icra_ve_iflas_hukuku_001,yargitay_0949_icra_ve_iflas_hukuku_004,yargitay_0949_icra_ve_iflas_hukuku_001,yargitay_0949_icra_ve_iflas_hukuku_001; yargit...,yargitay_0949_icra_ve_iflas_hukuku_004; yargit...,yargitay_0949_icra_ve_iflas_hukuku_001; yargit...,False,False,True,True,False,True
3,3,Türk Medenî Kanunu madde 879 kapsamında DÖRDÜN...,turkish_law_eski_4721_turk_medeni_kanunu_m879,turkish_law_eski_4721_turk_medeni_kanunu_m879,turkish_law_eski_4721_turk_medeni_kanunu_m879,turkish_law_eski_4721_turk_medeni_kanunu_m879,turkish_law_eski_4721_turk_medeni_kanunu_m879;...,turkish_law_eski_4721_turk_medeni_kanunu_m879;...,turkish_law_eski_4721_turk_medeni_kanunu_m879;...,True,True,True,True,True,True
4,4,Medeni Hukuk - Miras/Aile/Eşya/Kişiler alanınd...,oricon_medeni_000411,oricon_medeni_000411,oricon_medeni_000450,oricon_medeni_000411,oricon_medeni_000411; oricon_medeni_000361; or...,oricon_medeni_000450; oricon_medeni_000454; or...,oricon_medeni_000411; oricon_medeni_000361; or...,True,True,False,True,True,True


In [44]:
reranker_retrieval_summary_df = pd.DataFrame([
    {
        "method": "Hybrid Retrieval",
        "candidate_k": candidate_k,
        "final_k": final_k,
        "hit_at_1": reranker_retrieval_comparison_df["hybrid_hit_at_1"].mean(),
        "hit_at_5": reranker_retrieval_comparison_df["hybrid_hit_at_5"].mean()
    },
    {
        "method": "Hybrid + Base Turkish BGE Reranker",
        "candidate_k": candidate_k,
        "final_k": final_k,
        "hit_at_1": reranker_retrieval_comparison_df["base_rerank_hit_at_1"].mean(),
        "hit_at_5": reranker_retrieval_comparison_df["base_rerank_hit_at_5"].mean()
    },
    {
        "method": "Hybrid + Fine-tuned Turkish BGE Reranker",
        "candidate_k": candidate_k,
        "final_k": final_k,
        "hit_at_1": reranker_retrieval_comparison_df["finetuned_rerank_hit_at_1"].mean(),
        "hit_at_5": reranker_retrieval_comparison_df["finetuned_rerank_hit_at_5"].mean()
    }
])

reranker_retrieval_summary_df

,method,candidate_k,final_k,hit_at_1,hit_at_5
0,Hybrid Retrieval,20,5,0.77,0.89
1,Hybrid + Base Turkish BGE Reranker,20,5,0.84,0.94
2,Hybrid + Fine-tuned Turkish BGE Reranker,20,5,0.84,0.93


In [46]:
starlar_eval_df = pd.DataFrame(starlar_rag_eval_records).reset_index(drop=True)

candidate_k = 20
final_k = 5

comparison_rows_full = []

for i, row in tqdm(starlar_eval_df.iterrows(), total=len(starlar_eval_df)):
    query = row["query"]
    gold_chunk_ids = [str(x) for x in row["gold_chunk_ids"]]

    candidates = starlar_hybrid_retrieve_candidates(
        query,
        candidate_k=candidate_k,
        alpha=0.5
    )

    hybrid_top5 = candidates[:final_k]

    base_top5 = rerank_candidates(
        query=query,
        candidates=candidates,
        reranker=base_reranker_for_eval,
        top_k=final_k,
        batch_size=8
    )

    finetuned_top5 = rerank_candidates(
        query=query,
        candidates=candidates,
        reranker=finetuned_reranker,
        top_k=final_k,
        batch_size=8
    )

    hybrid_ids = [str(x["chunk_id"]) for x in hybrid_top5]
    base_ids = [str(x["chunk_id"]) for x in base_top5]
    finetuned_ids = [str(x["chunk_id"]) for x in finetuned_top5]

    comparison_rows_full.append({
        "index": i,
        "query": query,
        "gold_chunk_ids": "; ".join(gold_chunk_ids),

        "hybrid_hit_at_1": any(gold_id == hybrid_ids[0] for gold_id in gold_chunk_ids),
        "hybrid_hit_at_5": any(gold_id in hybrid_ids for gold_id in gold_chunk_ids),

        "base_rerank_hit_at_1": any(gold_id == base_ids[0] for gold_id in gold_chunk_ids),
        "base_rerank_hit_at_5": any(gold_id in base_ids for gold_id in gold_chunk_ids),

        "finetuned_rerank_hit_at_1": any(gold_id == finetuned_ids[0] for gold_id in gold_chunk_ids),
        "finetuned_rerank_hit_at_5": any(gold_id in finetuned_ids for gold_id in gold_chunk_ids),
    })

reranker_retrieval_comparison_full_df = pd.DataFrame(comparison_rows_full)

reranker_retrieval_summary_full_df = pd.DataFrame([
    {
        "method": "Hybrid Retrieval",
        "candidate_k": candidate_k,
        "final_k": final_k,
        "eval_sample_size": len(reranker_retrieval_comparison_full_df),
        "hit_at_1": reranker_retrieval_comparison_full_df["hybrid_hit_at_1"].mean(),
        "hit_at_5": reranker_retrieval_comparison_full_df["hybrid_hit_at_5"].mean()
    },
    {
        "method": "Hybrid + Base Turkish BGE Reranker",
        "candidate_k": candidate_k,
        "final_k": final_k,
        "eval_sample_size": len(reranker_retrieval_comparison_full_df),
        "hit_at_1": reranker_retrieval_comparison_full_df["base_rerank_hit_at_1"].mean(),
        "hit_at_5": reranker_retrieval_comparison_full_df["base_rerank_hit_at_5"].mean()
    },
    {
        "method": "Hybrid + Fine-tuned Turkish BGE Reranker",
        "candidate_k": candidate_k,
        "final_k": final_k,
        "eval_sample_size": len(reranker_retrieval_comparison_full_df),
        "hit_at_1": reranker_retrieval_comparison_full_df["finetuned_rerank_hit_at_1"].mean(),
        "hit_at_5": reranker_retrieval_comparison_full_df["finetuned_rerank_hit_at_5"].mean()
    }
])

reranker_retrieval_summary_full_df

100%|██████████| 1000/1000 [43:57<00:00,  2.64s/it]


,method,candidate_k,final_k,eval_sample_size,hit_at_1,hit_at_5
0,Hybrid Retrieval,20,5,1000,0.753,0.870
1,Hybrid + Base Turkish BGE Reranker,20,5,1000,0.813,0.912
2,Hybrid + Fine-tuned Turkish BGE Reranker,20,5,1000,0.803,0.914


In [47]:
reranker_retrieval_comparison_full_df.to_csv(
    f"{metrics_path}/finetuned_reranker_retrieval_comparison_full.csv",
    index=False,
    encoding="utf-8-sig"
)

reranker_retrieval_summary_full_df.to_csv(
    f"{metrics_path}/finetuned_reranker_retrieval_summary_full.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Full retrieval evaluation saved.")

Full retrieval evaluation saved.


## Reranker Fine-Tuning Conclusion

The Turkish BGE reranker was fine-tuned using the CENG493 Starlar reranker dataset. Pair-level validation performance improved strongly after fine-tuning. Validation AUC increased from 0.8386 to 0.9885, and best-threshold F1 increased from 0.6857 to 0.9311.

However, full retrieval-level evaluation on 1000 RAG queries showed a mixed result. The base Turkish BGE reranker achieved Hit@1 = 0.813 and Hit@5 = 0.912. The fine-tuned reranker achieved Hit@1 = 0.803 and Hit@5 = 0.914.

This means that fine-tuning improved relevance classification, but did not clearly outperform the base reranker in retrieval ranking. Therefore, the base Turkish BGE reranker remains the preferred reranker for the final RAG pipeline, while the fine-tuned reranker is reported as an experimental result.